# İşlemcide (CPU) çıkarım süresi: sıfır atışlı yerel yöntemler aynı donanımda

Bu defter makalede sıfır atışla kullanılan tüm yerel yöntemlerin (TF-IDF, statik kelime vektörleri, USE, cümle kodlayıcıları ve güncel çok dilli
kodlayıcılar) paylaşım başına çıkarım süresini **aynı işlemcide** ve aynı protokolle ölçer. Az örnekli sınıflandırıcılar (SetFit, dondurulmuş gömmeler üzerinde lojistik regresyon) ve ilçe çıkarımındaki NER modeli bu defterde ölçülmez. API çağrısı yapmaz, ücret doğurmaz.

**Çalıştırmadan önce:** *Çalışma zamanı → Çalışma zamanı türünü değiştir* → Donanım hızlandırıcı: **CPU** (GPU değil),
**Yüksek RAM** açık. Ardından *Çalışma zamanı → Tümünü çalıştır*. Yanlışlıkla GPU'lu çalışma zamanı seçilse de ölçüm işlemcide
yapılır, ancak işlemci farklı olacağından CPU çalışma zamanı tercih edilmelidir.

**Protokol:** her model için ihtiyaç vektörleri önceden hesaplanır (ölçülmez); üç ısınma çağrısından sonra makaledeki 30 rastgele
paylaşımın (tohum 42) her biri tek tek üç tur kodlanır ve 90 ölçümün ortancası alınır (tek paylaşım gecikmesi); ayrıca sabit
256 paylaşım 32'lik yığınlarla kodlanarak paylaşım başına süre hesaplanır (toplu kodlama). Modeller makaledeki ayarlarla
(görev talimatı, en fazla 512 belirteç, ihtiyaç metinleri) yüklenir. Doğrulama için 256 paylaşımdaki ilk tahminler de kaydedilir.

**Çıktılar:** `MyDrive/NLP_Disaster/runs/revizyon/results/revizyon2/` altında `cpu_gecikme.json`, `cpu_gecikme.csv` ve
`cpu_gecikme_tahminler.csv`. Her modelden sonra dosyalar güncellenir; oturum yarıda kesilirse yeniden çalıştırınca ölçülmüş
modeller atlanır. Makaledeki ölçümün çıktıları depoda `runs/revizyon/results/revizyon2/` altındadır.

In [ ]:
# 1. KURULUM VE DONANIM BİLGİSİ
import os, sys, subprocess, importlib
os.environ["CUDA_VISIBLE_DEVICES"] = ""          # ölçüm her durumda işlemcide
os.environ["TOKENIZERS_PARALLELISM"] = "false"

SEED = 42
N_TEK, N_TEKRAR, N_ISINMA = 30, 3, 3             # tek paylaşım: 30 paylaşım x 3 tur, 3 ısınma çağrısı
N_TOPLU, TOPLU_BOYUT = 256, 32                   # toplu kodlama: 256 paylaşım, 32'lik yığınlar
DRIVE_FOLDER = "NLP_Disaster"
PAPER_RUN = "20261016_1116"
REPO_RAW = "https://raw.githubusercontent.com/kurbankotan/disaster-needs-assessment-nlp/main"
EMB_TASK = "Given a social media post written after an earthquake, retrieve the relief item that the author needs most urgently"
MODEL_DIR_CANDIDATES = {
    "Word2Vec": ["MyDrive/Colab Notebooks/NLP Modeller/Word2Vec Files/word2vec-google-news-300.bin",
                 "MyDrive/NLP_Disaster/models/word2vec-google-news-300.bin"],
    "GloVe":    ["MyDrive/NLP_Disaster/models/glove.840B.300d.kv",
                 "MyDrive/Colab Notebooks/NLP Modeller/GloVe File/glove.840B.300d.txt",
                 "MyDrive/Colab Notebooks/NLP Modeller/GloVe File/glove.840B.300d-001.txt"],
    "FastText": ["MyDrive/Colab Notebooks/NLP Modeller/FastText Files/cc.en.300.bin",
                 "MyDrive/NLP_Disaster/models/cc.en.300.bin"],
    "USE":      ["MyDrive/Colab Notebooks/NLP Modeller/USE Files",
                 "MyDrive/NLP_Disaster/models/USE"],
}

def pip(*pk):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pk], check=False)

for mod, pk in [("sentence_transformers", "sentence-transformers>=3.0"), ("gensim", "gensim"), ("tensorflow_hub", "tensorflow-hub"),
                ("inflect", "inflect"), ("psutil", "psutil"), ("openpyxl", "openpyxl"), ("snowballstemmer", "snowballstemmer")]:
    try:
        importlib.import_module(mod)
    except ImportError:
        pip(pk)
try:
    import tensorflow  # noqa: F401  (USE için)
except ImportError:
    pip("tensorflow-cpu")
import transformers
from packaging.version import Version
if Version(transformers.__version__) < Version("4.51"):
    pip("-U", "transformers>=4.51")              # Qwen3-Embedding için
    print("transformers güncellendi: Çalışma zamanı > Oturumu yeniden başlat, sonra tümünü yeniden çalıştırın.")

import json, time, re, gc, random, platform, math, urllib.request, zipfile
import numpy as np, pandas as pd, psutil, torch
torch.set_num_threads(os.cpu_count())

try:
    from google.colab import drive, files
    drive.mount("/content/drive", force_remount=False)
    ROOT = "/content/drive"
except Exception:
    files = None
    ROOT = os.path.abspath("./drive_local")
OUT = os.path.join(ROOT, "MyDrive", DRIVE_FOLDER, "runs", "revizyon", "results", "revizyon2")
os.makedirs(OUT, exist_ok=True)
SCRATCH = "/content/cpu_scratch" if os.path.exists("/content") else os.path.abspath("./cpu_scratch")
os.makedirs(SCRATCH, exist_ok=True)

def cpu_model():
    try:
        for line in open("/proc/cpuinfo"):
            if line.startswith("model name"):
                return line.split(":", 1)[1].strip()
    except Exception:
        pass
    return platform.processor()

def surum(mod):
    try:
        return importlib.import_module(mod).__version__
    except Exception:
        return None

DONANIM = {"islemci": cpu_model(), "mantiksal_cekirdek": os.cpu_count(), "fiziksel_cekirdek": psutil.cpu_count(logical=False),
           "ram_gb": round(psutil.virtual_memory().total / 1e9, 1), "torch_is_parcacigi": torch.get_num_threads(),
           "gpu_kullanildi": bool(torch.cuda.is_available()), "platform": platform.platform(), "python": platform.python_version(),
           "lscpu": subprocess.run(["lscpu"], capture_output=True, text=True).stdout}
SURUMLER = {m: surum(m) for m in ["torch", "transformers", "sentence_transformers", "tensorflow", "tensorflow_hub", "gensim",
                                  "sklearn", "numpy"]}
PROTOKOL = {"tek_paylasim": f"{N_TEK} paylaşım x {N_TEKRAR} tur, {N_ISINMA} ısınma çağrısı, ortanca", "toplu": f"{N_TOPLU} paylaşım, yığın {TOPLU_BOYUT}",
            "olculen": "paylaşımın vektöre dönüştürülmesi (ön işleme hariç); ihtiyaç vektörleri önceden hesaplanır", "tohum": SEED}
print(json.dumps({k: v for k, v in DONANIM.items() if k != "lscpu"}, ensure_ascii=False, indent=1))
print(SURUMLER)
assert not DONANIM["gpu_kullanildi"], "GPU görünür durumda; ölçüm işlemcide olmalı"

In [ ]:
# 2. VERİ VE ÖN İŞLEME (makaledeki ön işlenmiş sütunlar kullanılır)
import nltk, inflect, snowballstemmer
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

def paper_file(rel):
    p = os.path.join(ROOT, "MyDrive", DRIVE_FOLDER, "runs", PAPER_RUN, rel)
    if os.path.exists(p):
        return p
    local = os.path.join(SCRATCH, os.path.basename(rel))
    if not os.path.exists(local):
        urllib.request.urlretrieve(f"{REPO_RAW}/runs/{PAPER_RUN}/{rel}", local)
    return local

df = pd.read_excel(paper_file("results/veri_seti_sentetik_islenmis.xlsx"))
need = pd.read_csv(paper_file("results/ihtiyac_listesi.csv"))
NEEDS_TR = need["tr"].tolist()
TR2EN = dict(zip(need["tr"], need["en"]))
EN_LABELS = list(dict.fromkeys(need["en"]))
GOLD = [{TR2EN[x] for x in (r.label1, r.label2, r.label3) if isinstance(x, str) and x in TR2EN} for r in df.itertuples()]
assert len(df) == 2000 and len(EN_LABELS) == 66, (len(df), len(EN_LABELS))

INFLECT = inflect.engine()
STOP_EN = set(stopwords.words("english"))
STEMMER = PorterStemmer()
LETTERS_RE = re.compile(r"[^a-zA-ZğĞçÇşŞüÜöÖıİ]")
LET_TR = re.compile(r"[^a-zA-ZğĞçÇşŞüÜöÖıİâÂîÎûÛ]")
STEM_TR = snowballstemmer.stemmer("turkish")
def numbers_to_words(t):
    return re.sub(r"\d+", lambda m: " " + (INFLECT.number_to_words(int(m.group())) if len(m.group()) < 12 else "") + " ", t)
def en_lower(s):
    return str(s).replace("İ", "i").lower()
def plain_text(t):
    return " ".join(en_lower(LETTERS_RE.sub(" ", numbers_to_words(t))).split()) if isinstance(t, str) else ""
def stem_text(t):
    return " ".join(STEMMER.stem(w) for w in plain_text(t).split() if w not in STOP_EN)
def tr_lower(s):
    return str(s).replace("İ", "i").replace("I", "ı").lower()
def plain_text_tr(t):
    return " ".join(tr_lower(LETTERS_RE.sub(" ", str(t))).split())
def stem_tr(t):
    return " ".join(STEM_TR.stemWords(" ".join(tr_lower(LET_TR.sub(" ", str(t))).split()).split()))

GIRDI = {
    "en_plain": (df["text_plain"].fillna("").astype(str).tolist(), [plain_text(n) or n.lower() for n in EN_LABELS], list(EN_LABELS)),
    "en_raw":   (df["English"].astype(str).tolist(), list(EN_LABELS), list(EN_LABELS)),
    "en_stem":  (df["text_stem"].fillna("").astype(str).tolist(), [stem_text(n) or n.lower() for n in EN_LABELS], list(EN_LABELS)),
    "tr_plain": (df["Message"].map(plain_text_tr).tolist(), [plain_text_tr(n) for n in NEEDS_TR], [TR2EN[n] for n in NEEDS_TR]),
    "tr_stem":  (df["Message"].map(stem_tr).tolist(), [stem_tr(n) for n in NEEDS_TR], [TR2EN[n] for n in NEEDS_TR]),
}
IDX_TEK = random.Random(SEED).sample(range(len(df)), N_TEK)
IDX_TOPLU = [int(i) for i in np.random.default_rng(SEED).permutation(len(df))[:N_TOPLU]]

# Ön işleme süresi (ham paylaşımdan modele girecek metne), aynı 30 paylaşımda
def _onisleme_ms(fn, metinler):
    for s in metinler[:N_ISINMA]:
        fn(s)
    v = []
    for _ in range(N_TEKRAR):
        for s in metinler:
            t0 = time.perf_counter(); fn(s); v.append((time.perf_counter() - t0) * 1000)
    return float(np.median(v))
_en = [df["English"].iat[i] for i in IDX_TEK]; _tr = [df["Message"].iat[i] for i in IDX_TEK]
ONISLEME = {"duz_metin_en": _onisleme_ms(plain_text, _en), "govde_en": _onisleme_ms(stem_text, _en),
            "duz_metin_tr": _onisleme_ms(plain_text_tr, _tr), "govde_tr_snowball": _onisleme_ms(stem_tr, _tr)}
print("Ön işleme (ms/paylaşım):", {k: round(v, 3) for k, v in ONISLEME.items()})

In [ ]:
# 3. MODEL SARMALAYICILARI (makaledeki ayarlar)
def find_existing(cands):
    for rel in cands:
        p = os.path.join(ROOT, rel)
        if os.path.exists(p):
            return p
    return None

class STModel:
    """sentence-transformers kodlayıcısı. prompt: paylaşımlara eklenen görev talimatı (ihtiyaç metinlerine eklenmez)."""
    def __init__(self, name, prompt="", maxlen=None, normalize=False, dtype=None, qwen=False):
        from sentence_transformers import SentenceTransformer
        kw = {}
        if dtype is not None:
            kw["model_kwargs"] = {"torch_dtype": dtype}
        if qwen:
            kw["tokenizer_kwargs"] = {"padding_side": "left"}
        self.m = SentenceTransformer(name, device="cpu", **kw)
        if maxlen:
            self.m.max_seq_length = min(int(self.m.max_seq_length or maxlen), maxlen)
        self.prompt = prompt.replace("{task}", EMB_TASK) if prompt else ""
        self.normalize = normalize
        self.kaynak = f"sentence-transformers:{name}"
        self.parametre = int(sum(p.numel() for p in self.m.parameters()))
        self.boyut = int(self.m.get_sentence_embedding_dimension() or 0)
        self.dtype = str(next(self.m.parameters()).dtype).replace("torch.", "")
    def _e(self, texts, bs):
        return np.asarray(self.m.encode(list(texts), batch_size=bs, convert_to_numpy=True, show_progress_bar=False,
                                        normalize_embeddings=self.normalize), dtype=np.float32)
    def sorgu(self, texts, bs=TOPLU_BOYUT):
        return self._e([self.prompt + t for t in texts] if self.prompt else texts, bs)
    def belge(self, texts):
        return self._e(texts, TOPLU_BOYUT)

class KVModel:
    """Kelime vektörlerinin ortalaması (Word2Vec/GloVe: sözlükteki kelimeler; FastText: alt kelimelerle, birim normlu)."""
    def __init__(self, kv, kaynak, fasttext=False):
        self.kv, self.kaynak, self.fasttext = kv, kaynak, fasttext
        self.boyut = int(kv.vector_size); self.parametre = int(len(kv.key_to_index) * kv.vector_size); self.dtype = "float32"
        if fasttext and hasattr(kv, "vectors_ngrams"):
            self.parametre += int(np.prod(kv.vectors_ngrams.shape))
    def _vec(self, text):
        words = text.split()
        if self.fasttext:
            vs = [v for v in (self.kv.get_vector(w, norm=True) for w in words) if np.isfinite(v).all() and np.linalg.norm(v) > 0]
        else:
            vs = [self.kv[w] for w in words if w in self.kv.key_to_index]
        return np.mean(vs, axis=0) if vs else np.zeros(self.boyut, dtype=np.float32)
    def sorgu(self, texts, bs=None):
        return np.vstack([self._vec(t) for t in texts]).astype(np.float32)
    belge = sorgu

class USEModel:
    def __init__(self, model, kaynak):
        self.m, self.kaynak, self.dtype = model, kaynak, "float32"
        try:
            self.parametre = int(sum(int(np.prod(v.shape)) for v in model.variables))
        except Exception:
            self.parametre = None
        self.boyut = int(self.sorgu(["test"]).shape[1])
    def sorgu(self, texts, bs=256):
        texts = list(texts)
        return np.vstack([self.m(texts[i:i + 256]).numpy() for i in range(0, len(texts), 256)]).astype(np.float32)
    belge = sorgu

class TfidfModel:
    def __init__(self, tum_metinler, ihtiyaclar, kaynak):
        from sklearn.feature_extraction.text import TfidfVectorizer
        self.vec = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, min_df=1).fit(list(tum_metinler) + list(ihtiyaclar))
        self.kaynak, self.dtype = kaynak, "float32"
        self.boyut = len(self.vec.vocabulary_); self.parametre = 0
    def sorgu(self, texts, bs=None):
        return self.vec.transform(list(texts)).toarray().astype(np.float32)
    belge = sorgu

def glove_txt_to_kv(txt_path, dim=300):
    from gensim.models import KeyedVectors
    n = sum(1 for _ in open(txt_path, "rb"))
    words, mat, seen = [], np.empty((n, dim), dtype=np.float32), set()
    with open(txt_path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            parts = line.rstrip("\n").split(" ")
            if len(parts) <= dim:
                continue
            w = " ".join(parts[:-dim])
            if w in seen:
                continue
            try:
                vec = np.asarray(parts[-dim:], dtype=np.float32)
            except ValueError:
                continue
            seen.add(w); words.append(w); mat[len(words) - 1] = vec
    kv = KeyedVectors(vector_size=dim); kv.add_vectors(words, mat[:len(words)])
    return kv

def yukle(tur, a):
    if tur == "tfidf":
        metin, ihtiyac, _ = GIRDI[a["girdi"]]
        return TfidfModel(metin, ihtiyac, a["kaynak"])
    if tur == "st":
        return STModel(a["name"], prompt=a.get("prompt", ""), maxlen=a.get("maxlen"), normalize=a.get("normalize", False),
                       dtype=a.get("dtype", torch.float32), qwen="qwen" in a["name"].lower())
    from gensim.models import KeyedVectors
    if tur == "w2v":
        p = find_existing(MODEL_DIR_CANDIDATES["Word2Vec"])
        if p:
            return KVModel(KeyedVectors.load_word2vec_format(p, binary=True), f"word2vec-google-news-300 ({p})")
        import gensim.downloader as gapi
        return KVModel(gapi.load("word2vec-google-news-300"), "word2vec-google-news-300 (gensim-data)")
    if tur == "glove":
        p = find_existing(MODEL_DIR_CANDIDATES["GloVe"])
        if p and p.endswith(".kv"):
            return KVModel(KeyedVectors.load(p, mmap="r"), f"glove.840B.300d ({p})")
        if not p:
            zp = os.path.join(SCRATCH, "glove.840B.300d.zip")
            if not os.path.exists(zp):
                urllib.request.urlretrieve("https://nlp.stanford.edu/data/glove.840B.300d.zip", zp)
            with zipfile.ZipFile(zp) as z:
                z.extractall(SCRATCH)
            p = os.path.join(SCRATCH, "glove.840B.300d.txt")
        return KVModel(glove_txt_to_kv(p), f"glove.840B.300d ({p})")
    if tur == "fasttext":
        from gensim.models.fasttext import load_facebook_vectors
        p = find_existing(MODEL_DIR_CANDIDATES["FastText"])
        if not p:
            import gzip, shutil
            gz = os.path.join(SCRATCH, "cc.en.300.bin.gz")
            if not os.path.exists(gz):
                urllib.request.urlretrieve("https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.en.300.bin.gz", gz)
            p = os.path.join(SCRATCH, "cc.en.300.bin")
            with gzip.open(gz, "rb") as fi, open(p, "wb") as fo:
                shutil.copyfileobj(fi, fo, length=64 * 1024 * 1024)
        return KVModel(load_facebook_vectors(p), f"cc.en.300.bin ({p})", fasttext=True)
    if tur == "use":
        import tensorflow as tf, tensorflow_hub as hub
        p = find_existing(MODEL_DIR_CANDIDATES["USE"])
        for src in ([p] if p else []) + ["https://tfhub.dev/google/universal-sentence-encoder/4",
                                         "https://www.kaggle.com/models/google/universal-sentence-encoder/TensorFlow2/universal-sentence-encoder/2"]:
            try:
                return USEModel(hub.load(src), f"universal-sentence-encoder/4 ({src})")
            except Exception as e:
                print("USE yüklenemedi:", src, repr(e)[:200])
        raise RuntimeError("USE yüklenemedi")
    raise ValueError(tur)

# Tüm sinir ağı kodlayıcıları işlemcide tam duyarlılıkla (float32) çalıştırılır. 4 milyar parametreli model float32 ile
# ~16 GB bellek ister; bellek 30 GB'tan azsa bu model bf16 ile yüklenir ve bu durum çıktıya yazılır.
QWEN4_DTYPE = torch.float32 if DONANIM["ram_gb"] >= 30 else torch.bfloat16
MODELLER = [
    ("TF-IDF (gövde)",       "tfidf", {"girdi": "en_stem", "kaynak": "TF-IDF (sklearn, 1-2 gram, sublinear tf), Porter gövdesi"}),
    ("TF-IDF (Snowball, TR)", "tfidf", {"girdi": "tr_stem", "kaynak": "TF-IDF (sklearn, 1-2 gram, sublinear tf), Snowball Türkçe gövdesi"}),
    ("USE",                  "use",   {"girdi": "en_plain"}),
    ("SBERT-mpnet",          "st",    {"girdi": "en_plain", "name": "all-mpnet-base-v2"}),
    ("RoBERTa-güncel",       "st",    {"girdi": "en_plain", "name": "all-distilroberta-v1"}),
    ("ALBERT-paraphrase",    "st",    {"girdi": "en_plain", "name": "paraphrase-albert-small-v2"}),
    ("BERT (NLI)",           "st",    {"girdi": "en_plain", "name": "bert-base-nli-mean-tokens"}),
    ("RoBERTa (NLI)",        "st",    {"girdi": "en_plain", "name": "roberta-base-nli-mean-tokens"}),
    ("BERT-raw",             "st",    {"girdi": "en_plain", "name": "bert-base-uncased"}),
    ("RoBERTa-raw",          "st",    {"girdi": "en_plain", "name": "roberta-base"}),
    ("ALBERT",               "st",    {"girdi": "en_plain", "name": "albert-base-v2"}),
    ("SBERT çok dilli (TR)", "st",    {"girdi": "tr_plain", "name": "paraphrase-multilingual-MiniLM-L12-v2"}),
    ("BERTurk-STS (TR)",     "st",    {"girdi": "tr_plain", "name": "emrecan/bert-base-turkish-cased-mean-nli-stsb-tr"}),
    ("mE5-large-instruct",   "st",    {"girdi": "en_plain", "name": "intfloat/multilingual-e5-large-instruct", "prompt": "Instruct: {task}\nQuery: ", "maxlen": 512, "normalize": True}),
    ("BGE-M3",               "st",    {"girdi": "en_raw",   "name": "BAAI/bge-m3", "maxlen": 512, "normalize": True}),
    ("Qwen3-Embedding-0.6B", "st",    {"girdi": "en_plain", "name": "Qwen/Qwen3-Embedding-0.6B", "prompt": "Instruct: {task}\nQuery:", "maxlen": 512, "normalize": True}),
    ("Word2Vec",             "w2v",   {"girdi": "en_plain"}),
    ("FastText",             "fasttext", {"girdi": "en_plain"}),
    ("GloVe",                "glove", {"girdi": "en_plain"}),
    ("Qwen3-Embedding-4B",   "st",    {"girdi": "en_plain", "name": "Qwen/Qwen3-Embedding-4B", "prompt": "Instruct: {task}\nQuery:", "maxlen": 512, "normalize": True, "dtype": QWEN4_DTYPE}),
]
_SEC = os.environ.get("CPU_TEST_MODELS")          # yalnız yerel deneme için; Colab'da boş bırakın
if _SEC:
    MODELLER = [m for m in MODELLER if m[0] in _SEC.split("|")]
print(len(MODELLER), "yöntem ölçülecek")

In [ ]:
# 4. ÖLÇÜM (her modelden sonra sonuçlar Drive'a yazılır; yeniden çalıştırınca ölçülmüş modeller atlanır)
JSON_P, CSV_P, PRED_P = (os.path.join(OUT, f) for f in ("cpu_gecikme.json", "cpu_gecikme.csv", "cpu_gecikme_tahminler.csv"))
SONUC = json.load(open(JSON_P, encoding="utf-8")) if os.path.exists(JSON_P) else {}
if SONUC.get("donanim", {}).get("islemci") not in (None, DONANIM["islemci"]) or SONUC.get("donanim", {}).get("mantiksal_cekirdek") not in (None, DONANIM["mantiksal_cekirdek"]):
    print("UYARI: önceki ölçümler farklı bir işlemcide yapılmış; aynı donanım koşulu için hepsi yeniden ölçülecek.")
    SONUC = {}
SONUC.update({"donanim": DONANIM, "surumler": SURUMLER, "protokol": PROTOKOL, "on_isleme_ms": ONISLEME})
SONUC.setdefault("modeller", {})
TAHMIN = pd.read_csv(PRED_P) if os.path.exists(PRED_P) else pd.DataFrame({"id": df["id"].iloc[IDX_TOPLU].values})

def kaydet():
    json.dump(SONUC, open(JSON_P, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    rows = [{"model": k, **{kk: vv for kk, vv in v.items() if kk != "hata_ayrinti"}} for k, v in SONUC["modeller"].items()]
    pd.DataFrame(rows).to_csv(CSV_P, index=False, encoding="utf-8-sig")
    TAHMIN.to_csv(PRED_P, index=False, encoding="utf-8-sig")

def olc(ad, tur, a):
    t0 = time.perf_counter(); m = yukle(tur, a); t_yukle = time.perf_counter() - t0
    metin, ihtiyac, etiket = GIRDI[a["girdi"]]
    Nv = m.belge(ihtiyac); Nv = Nv / (np.linalg.norm(Nv, axis=1, keepdims=True) + 1e-12)
    tek = [metin[i] for i in IDX_TEK]
    for s in tek[:N_ISINMA]:
        m.sorgu([s], 1)
    v = []
    for _ in range(N_TEKRAR):
        for s in tek:
            t1 = time.perf_counter(); m.sorgu([s], 1); v.append((time.perf_counter() - t1) * 1000)
    toplu = [metin[i] for i in IDX_TOPLU]
    t1 = time.perf_counter(); M = m.sorgu(toplu, TOPLU_BOYUT); t_toplu = (time.perf_counter() - t1) / len(toplu) * 1000
    M = M / (np.linalg.norm(M, axis=1, keepdims=True) + 1e-12)
    top1 = [etiket[j] for j in np.argmax(M @ Nv.T, axis=1)]
    hit = float(np.mean([t in GOLD[i] for t, i in zip(top1, IDX_TOPLU)]))
    TAHMIN[ad] = top1
    r = {"kaynak": m.kaynak, "girdi": a["girdi"], "dtype": getattr(m, "dtype", None), "parametre": m.parametre, "boyut": m.boyut,
         "yukleme_sn": round(t_yukle, 1), "tek_ms_ortanca": float(np.median(v)), "tek_ms_p90": float(np.percentile(v, 90)),
         "tek_ms_tur_ortancalari": [float(np.median(v[k * N_TEK:(k + 1) * N_TEK])) for k in range(N_TEKRAR)],
         "toplu_ms_paylasim": float(t_toplu), "hit1_256": hit}
    del m; gc.collect()
    return r

for ad, tur, a in MODELLER:
    if ad in SONUC["modeller"] and "tek_ms_ortanca" in SONUC["modeller"][ad]:
        print(f"{ad}: önceden ölçülmüş, atlanıyor"); continue
    print(f"{ad} ölçülüyor...", flush=True)
    try:
        r = olc(ad, tur, a)
        SONUC["modeller"][ad] = r
        print(f"  tek paylaşım {r['tek_ms_ortanca']:.3f} ms | toplu {r['toplu_ms_paylasim']:.3f} ms/paylaşım | yükleme {r['yukleme_sn']} sn | hit@1 (256) {r['hit1_256']:.3f}")
    except Exception as e:
        SONUC["modeller"][ad] = {"hata": repr(e)[:500]}
        print("  HATA:", repr(e)[:300])
    kaydet()

In [ ]:
# 5. ÖZET VE İNDİRME
ozet = pd.DataFrame([{"Yöntem": k, "Tek paylaşım (ms)": v.get("tek_ms_ortanca"), "Toplu (ms/paylaşım)": v.get("toplu_ms_paylasim"),
                      "dtype": v.get("dtype"), "Hata": v.get("hata")} for k, v in SONUC["modeller"].items()])
print(DONANIM["islemci"], "|", DONANIM["mantiksal_cekirdek"], "mantıksal çekirdek |", DONANIM["ram_gb"], "GB RAM")
print(ozet.to_string(index=False))
if files is not None:
    for p in (JSON_P, CSV_P, PRED_P):
        try:
            files.download(p)
        except Exception as e:
            print("İndirilemedi:", p, e)
print("Dosyalar:", OUT)